# landing_eda_transparencia — EDA Portal da Transparência (CEIS e CNEP)

> **Exploration only — not part of the production pipeline.**

## Purpose
Understand structure, quality, and anomalies in the CEIS and CNEP raw JSON files
before defining and validating the Bronze schema.

## Central question
*"What is the schema of CEIS and CNEP, how do they differ,
what nested objects exist, and what design decisions does Bronze need?"*

## Engine
DuckDB — same engine used in Bronze pipeline. No pandas.

## Steps
1. Imports and configuration
2. File inventory
3. Schema discovery — CEIS
4. Schema discovery — CNEP
5. Schema comparison — CEIS vs CNEP
6. Key field analysis
7. Nested objects inspection
8. Inconsistencies and design decisions
9. Dynamic summary


## Step 1 — Imports and configuration

In [ ]:
import sys
from pathlib import Path

# --- Resolve root ------------------------------------------------------------
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent.parent  # eda/ → notebooks/ → fornecedor360-local/
except NameError:
    _root_candidate = Path.cwd()
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
        if (candidate / "utils" / "paths.py").exists():
            _root_candidate = candidate
            break

sys.path.insert(0, str(_root_candidate / "utils"))

from paths        import get_project_root, to_sql_path
from duckdb_utils import get_connection

PROJECT_ROOT = get_project_root()
con = get_connection()  # in-memory — EDA never writes

TRANSP_ROOT = PROJECT_ROOT / "data" / "raw" / "portal_transparencia"
CEIS_PATH   = to_sql_path(TRANSP_ROOT / "ceis.json")
CNEP_PATH   = to_sql_path(TRANSP_ROOT / "cnep.json")

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"TRANSP_ROOT  : {TRANSP_ROOT}")
print(f"CEIS exists  : {(TRANSP_ROOT / 'ceis.json').exists()}")
print(f"CNEP exists  : {(TRANSP_ROOT / 'cnep.json').exists()}")

## Step 2 — File inventory

**Question:** *How many records are in CEIS and CNEP?
What are the file sizes? Are they consistent with a full historical reload?*


In [ ]:
print("=== File Inventory ===")
print()
print(f"{'file':<15} {'size_mb':>8} {'records':>10}")
print("-" * 37)

for name, path_sql, fpath in [
    ("ceis.json", CEIS_PATH, TRANSP_ROOT / "ceis.json"),
    ("cnep.json", CNEP_PATH, TRANSP_ROOT / "cnep.json"),
]:
    if not fpath.exists():
        print(f"{name:<15} [MISSING]")
        continue
    size_mb = fpath.stat().st_size / 1e6
    records = con.execute(f"SELECT COUNT(*) FROM read_json_auto('{path_sql}')").fetchone()[0]
    print(f"{name:<15} {size_mb:>8.1f} {records:>10,}")


## Step 3 — Schema discovery — CEIS

**Question:** *What columns does CEIS have? Which are nested?
What are the null rates? What field contains the CNPJ?*


In [ ]:
print("=== Schema Discovery — CEIS ===")
print()

ceis_schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{CEIS_PATH}') LIMIT 0").fetchall()
ceis_total  = con.execute(f"SELECT COUNT(*) FROM read_json_auto('{CEIS_PATH}')").fetchone()[0]

print(f"Records : {ceis_total:,}  |  Columns : {len(ceis_schema)}")
print()
print(f"{'column':<50} {'type':<20} {'nulls':>8} {'null%':>6}")
print("-" * 90)

for col, dtype, *_ in ceis_schema:
    if any(t in dtype.upper() for t in ["STRUCT", "LIST", "MAP"]):
        print(f"{col:<50} {dtype[:20]:<20} {'(nested)':>8}")
        continue
    nulls = con.execute(
        f"SELECT COUNT(*) FROM read_json_auto('{CEIS_PATH}') WHERE \"{col}\" IS NULL"
    ).fetchone()[0]
    pct = nulls / ceis_total * 100 if ceis_total else 0
    print(f"{col:<50} {dtype[:20]:<20} {nulls:>8,} {pct:>5.1f}%")


## Step 4 — Schema discovery — CNEP

**Question:** *What columns does CNEP have? How does it differ from CEIS?
Does it have valorMulta that CEIS does not?*


In [ ]:
print("=== Schema Discovery — CNEP ===")
print()

cnep_schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{CNEP_PATH}') LIMIT 0").fetchall()
cnep_total  = con.execute(f"SELECT COUNT(*) FROM read_json_auto('{CNEP_PATH}')").fetchone()[0]

print(f"Records : {cnep_total:,}  |  Columns : {len(cnep_schema)}")
print()
print(f"{'column':<50} {'type':<20} {'nulls':>8} {'null%':>6}")
print("-" * 90)

for col, dtype, *_ in cnep_schema:
    if any(t in dtype.upper() for t in ["STRUCT", "LIST", "MAP"]):
        print(f"{col:<50} {dtype[:20]:<20} {'(nested)':>8}")
        continue
    nulls = con.execute(
        f"SELECT COUNT(*) FROM read_json_auto('{CNEP_PATH}') WHERE \"{col}\" IS NULL"
    ).fetchone()[0]
    pct = nulls / cnep_total * 100 if cnep_total else 0
    print(f"{col:<50} {dtype[:20]:<20} {nulls:>8,} {pct:>5.1f}%")


## Step 5 — Schema comparison — CEIS vs CNEP

**Question:** *Which fields are shared between CEIS and CNEP?
Which are exclusive to one source? Does this justify two separate Bronze tables?*


In [ ]:
print("=== Schema Comparison — CEIS vs CNEP ===")
print()

ceis_cols = {col: dtype for col, dtype, *_ in ceis_schema}
cnep_cols = {col: dtype for col, dtype, *_ in cnep_schema}

only_ceis = sorted(set(ceis_cols) - set(cnep_cols))
only_cnep = sorted(set(cnep_cols) - set(ceis_cols))
shared    = sorted(set(ceis_cols) & set(cnep_cols))

print(f"Shared columns  : {len(shared)}")
print(f"Only in CEIS    : {len(only_ceis)} → {only_ceis}")
print(f"Only in CNEP    : {len(only_cnep)} → {only_cnep}")
print()
print("Type match on shared columns:")
for col in shared:
    ct, nt = ceis_cols[col], cnep_cols[col]
    match = "OK" if ct == nt else f"MISMATCH ({ct} vs {nt})"
    print(f"  {col:<45} {match}")


## Step 6 — Key field analysis

**Question:** *Where is the CNPJ in these records? Is it nested inside pessoa?
What is the format (with or without mask)? Are there PF records to filter?*


In [ ]:
print("=== Key Field Analysis ===")
print()

print("--- pessoa.cnpjFormatado (CNPJ with mask XX.XXX.XXX/XXXX-XX) ---")
for label, path_sql, total in [("CEIS", CEIS_PATH, ceis_total), ("CNEP", CNEP_PATH, cnep_total)]:
    result = con.execute(f"""
        SELECT
            COUNT(*) FILTER (WHERE pessoa.cnpjFormatado IS NULL)     AS nulls,
            COUNT(*) FILTER (WHERE pessoa.cnpjFormatado = '')        AS empty_string,
            COUNT(DISTINCT pessoa.cnpjFormatado)                     AS distinct_cnpj,
            COUNT(*) FILTER (WHERE pessoa.tipo = 'Pessoa Física')    AS pessoa_fisica
        FROM read_json_auto('{path_sql}')
    """).fetchone()
    print(f"  {label}: nulls={result[0]:,}  empty={result[1]:,}  distinct={result[2]:,}  PF={result[3]:,}")

print()
print("--- Sample CNPJ values (to confirm format) ---")
con.execute(f"""
    SELECT pessoa.cnpjFormatado AS cnpj_raw, pessoa.tipo AS tipo
    FROM read_json_auto('{CEIS_PATH}')
    WHERE pessoa.cnpjFormatado IS NOT NULL AND pessoa.cnpjFormatado != ''
    LIMIT 5
""").df()

print()
print("--- dataInicioSancao format ---")
con.execute(f"SELECT dataInicioSancao FROM read_json_auto('{CEIS_PATH}') LIMIT 3").df()
print("[NOTE] Format: DD/MM/YYYY (Brazilian) — Silver: TRY_STRPTIME(col, '%d/%m/%Y')")


## Step 7 — Nested objects inspection

**Question:** *Which columns are STRUCT or LIST in CEIS?
How does the `pessoa` object look? What about `fundamentacao` (LIST)?*


In [ ]:
print("=== Nested Objects — CEIS ===")
print()

nested = [(col, dtype) for col, dtype, *_ in ceis_schema
          if any(t in dtype.upper() for t in ["STRUCT", "LIST", "MAP"])]
print(f"Nested columns: {[c for c, _ in nested]}")
print()

for col, dtype in nested:
    null_count = con.execute(
        f"SELECT COUNT(*) FROM read_json_auto('{CEIS_PATH}') WHERE \"{col}\" IS NULL"
    ).fetchone()[0]
    print(f"--- {col} ---")
    print(f"  Type        : {dtype[:100]}")
    print(f"  Null records: {null_count:,} / {ceis_total:,} ({null_count/ceis_total*100:.1f}%)")
    try:
        sample = con.execute(
            f"SELECT CAST(\"{col}\" AS VARCHAR) FROM read_json_auto('{CEIS_PATH}') "
            f"WHERE \"{col}\" IS NOT NULL LIMIT 1"
        ).fetchone()
        if sample:
            print(f"  Sample      : {str(sample[0])[:150]}")
    except Exception as e:
        print(f"  Sample error: {e}")
    print()


## Step 8 — Inconsistencies and design decisions

**Question:** *What anomalies require explicit handling in Bronze?
What are the key decisions made and their rationale?*


In [ ]:
print("=== Inconsistencies and Design Decisions ===")
print()

print("1. CNPJ nested in pessoa.cnpjFormatado — with mask XX.XXX.XXX/XXXX-XX")
pf_ceis = con.execute(
    f"SELECT COUNT(*) FROM read_json_auto('{CEIS_PATH}') WHERE pessoa.tipo = 'Pessoa Física'"
).fetchone()[0]
pf_cnep = con.execute(
    f"SELECT COUNT(*) FROM read_json_auto('{CNEP_PATH}') WHERE pessoa.tipo = 'Pessoa Física'"
).fetchone()[0]
print(f"   PF records: CEIS={pf_ceis:,}  CNEP={pf_cnep:,}")
print("   Bronze: REGEXP_REPLACE(pessoa.cnpjFormatado, '[^0-9]', '', 'g') → 14-digit CNPJ")
print("   Silver: filter pessoa.tipo = 'Pessoa Jurídica'")
print()

print("2. Date format — DD/MM/YYYY (Brazilian), NOT ISO 8601")
print("   Bronze: keep as STRING. Silver: TRY_STRPTIME(col, '%d/%m/%Y')::DATE")
print()

print("3. valorMulta — CNEP only (Brazilian decimal format: comma separator)")
empty_cnep = con.execute(
    f"SELECT COUNT(*) FROM read_json_auto('{CNEP_PATH}') WHERE valorMulta = 'Sem informação'"
).fetchone()[0]
print(f"   'Sem informação' sentinel: {empty_cnep:,} rows in CNEP")
print("   Bronze: keep as STRING. Silver: REPLACE(',','.') then TRY_CAST to DECIMAL")
print()

print("4. fundamentacao — LIST type")
fund_type = ceis_cols.get("fundamentacao", "NOT FOUND")
print(f"   DuckDB type: {fund_type}")
print("   Bronze: to_json(fundamentacao)::VARCHAR — serialize LIST to JSON string")
print()

print("5. Full reload strategy — both files are full historical base")
print("   Bronze: TRUNCATE + full reload on every run")
print()

print("6. Duplicate check on id")
for label, path_sql, total in [("CEIS", CEIS_PATH, ceis_total), ("CNEP", CNEP_PATH, cnep_total)]:
    dup = con.execute(
        f"SELECT COUNT(*) - COUNT(DISTINCT id) FROM read_json_auto('{path_sql}')"
    ).fetchone()[0]
    print(f"   {label} duplicates on id: {dup:,}")


## Step 9 — Dynamic summary

**Question:** *What have I learned? What does Bronze depend on?*


In [ ]:
from IPython.display import display, Markdown

only_ceis_str = ", ".join(only_ceis) if only_ceis else "none"
only_cnep_str = ", ".join(only_cnep) if only_cnep else "none"

summary = f"""
## Portal da Transparência — EDA Summary

| | CEIS | CNEP |
|---|---|---|
| Records | {ceis_total:,} | {cnep_total:,} |
| Columns | {len(ceis_schema)} | {len(cnep_schema)} |
| Shared columns | {len(shared)} | {len(shared)} |
| Exclusive columns | {only_ceis_str} | {only_cnep_str} |

### Key Bronze decisions
| Decision | Reason |
|---|---|
| Two separate Bronze tables | Schemas differ — `valorMulta` is CNEP-only |
| All scalar columns as STRING | JSON source, Brazilian dates, comma decimal |
| `pessoa.cnpjFormatado` → remove mask | Bronze: REGEXP_REPLACE. Silver: 14-digit clean CNPJ |
| `fundamentacao` → `to_json()` → STRING | LIST type — serialize; explode in Silver if needed |
| Date as STRING in Bronze | Brazilian format DD/MM/YYYY — Silver parses |
| `valorMulta` as STRING in Bronze | Brazilian decimal + 'Sem informação' sentinel |
| Full truncate-and-reload | Source is full historical base — no incremental key |
"""

display(Markdown(summary))
